# Concrete Crack Detection Pipeline

This notebook demonstrates a modular, reproducible workflow for concrete crack image classification and segmentation using PyTorch. Each section elaborates on the features and design of the model modules, with runnable code and explanations.

## 1. Import Required Libraries

We begin by importing all necessary libraries for data processing, model building, training, and evaluation. Each library serves a specific purpose in the pipeline, from handling images to building neural networks.

In [ ]:
# Data handling and processing
import os
import numpy as np
import pandas as pd
from PIL import Image

# PyTorch core
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

# Computer vision and augmentation
import torchvision
from torchvision import transforms
import albumentations as A
from albumentations.pytorch import ToTensorV2

# Metrics and utilities
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

# Project modules
from src import data, models, losses, metrics, viz, config, train_utils

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## 2. Overview of Model Modules

This pipeline is organized into modular components, each responsible for a key part of the workflow:

- **src/data.py**: Data loading, augmentation, and stratified splitting for classification and segmentation.
- **src/models.py**: Model architectures for classification (ResNet/EfficientNet) and segmentation (U-Net).
- **src/losses.py**: Loss functions including BCE, Dice, BCE+Dice, and Focal loss.
- **src/metrics.py**: Metrics for evaluation (F1, ROC-AUC, Dice, IoU, etc.).
- **src/viz.py**: Visualization utilities for learning curves, batch samples, and mask overlays.
- **src/config.py**: YAML-based configuration loader and saver.
- **src/train_utils.py**: Utilities for reproducibility, checkpointing, and seed setting.

Each section below will elaborate on these modules and demonstrate their use.

## 3. Module 1: Data Preprocessing

Data preprocessing is crucial for robust model performance. This module handles:
- Loading images and (optionally) segmentation masks
- Stratified train/val/test splitting
- Data augmentation (random crops, flips, color jitter, etc.)
- Normalization and tensor conversion

Below, we demonstrate how to use the data module for both classification and segmentation tasks.

In [ ]:
# Example: Prepare classification dataset and dataloaders
from src.data import get_classification_filepaths, stratified_split, ConcreteClassificationDataset
from torchvision import transforms

# Set data root
DATA_ROOT = 'concrete-crack-images-for-classification'

# Get filepaths and labels
filepaths, labels = get_classification_filepaths(DATA_ROOT)

# Stratified split
splits = stratified_split(filepaths, labels, val_size=0.15, test_size=0.15, seed=42)
train_files, val_files, test_files = splits['train'], splits['val'], splits['test']
train_labels = [labels[filepaths.index(f)] for f in train_files]
val_labels = [labels[filepaths.index(f)] for f in val_files]
test_labels = [labels[filepaths.index(f)] for f in test_files]

# Define augmentations
train_transform = transforms.Compose([
    transforms.Resize((227, 227)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
val_transform = transforms.Compose([
    transforms.Resize((227, 227)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

# Create datasets
train_ds = ConcreteClassificationDataset(train_files, train_labels, transform=train_transform)
val_ds = ConcreteClassificationDataset(val_files, val_labels, transform=val_transform)
test_ds = ConcreteClassificationDataset(test_files, test_labels, transform=val_transform)

# Create dataloaders
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, num_workers=4)
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False, num_workers=4)
test_loader = DataLoader(test_ds, batch_size=64, shuffle=False, num_workers=4)

print(f"Train: {len(train_ds)}, Val: {len(val_ds)}, Test: {len(test_ds)}")

### Segmentation Data (Optional)

If segmentation masks are available, the data module can also handle paired image-mask loading and augmentation. Uncomment and adapt the following code if you have segmentation data.

In [ ]:
# Example: Prepare segmentation dataset and dataloaders (if masks exist)
# from src.data import get_segmentation_filepaths, ConcreteSegmentationDataset
# img_paths, mask_paths = get_segmentation_filepaths(DATA_ROOT)
# seg_transform = A.Compose([
#     A.Resize(256, 256),
#     A.HorizontalFlip(p=0.5),
#     A.Normalize(),
#     ToTensorV2(),
# ])
# seg_ds = ConcreteSegmentationDataset(img_paths, mask_paths, transform=seg_transform)
# seg_loader = DataLoader(seg_ds, batch_size=8, shuffle=True, num_workers=4)
# print(f"Segmentation samples: {len(seg_ds)}")

## 4. Module 2: Model Architecture

This module defines the neural network architectures for both classification and segmentation tasks:
- **Classification**: Uses a ResNet/EfficientNet backbone with optional dropout and custom head.
- **Segmentation**: Implements a U-Net architecture for pixel-wise mask prediction.

The models are designed for flexibility and can be easily extended or fine-tuned.

In [ ]:
# Example: Define classification model
from src.models import ClassificationModel

model = ClassificationModel()
model = model.to(device)
print(model)

In [ ]:
# Example: Define segmentation model (if masks exist)
# from src.models import UNet
# seg_model = UNet()
# seg_model = seg_model.to(device)
# print(seg_model)

## 5. Module 3: Training the Model

The training module handles the full training loop, including:
- Loss function selection (BCE, Dice, Focal, etc.)
- Optimizer and learning rate scheduling (AdamW, ReduceLROnPlateau)
- Early stopping and checkpointing
- Mixed precision (AMP) and gradient clipping

Below is an example of how to train the classification model. Segmentation training is similar, with appropriate loss and metrics.

In [ ]:
# Example: Training loop for classification
from src.losses import bce_loss
from src.metrics import classification_metrics
from src.train_utils import set_seed

set_seed(42)
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', patience=3, factor=0.5)

num_epochs = 3  # For demonstration; increase for real training
for epoch in range(num_epochs):
    model.train()
    train_loss = 0
    for imgs, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
        imgs, labels = imgs.to(device), labels.float().to(device)
        optimizer.zero_grad()
        logits = model(imgs)
        loss = bce_loss(logits, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * imgs.size(0)
    train_loss /= len(train_loader.dataset)
    print(f"Train Loss: {train_loss:.4f}")
    # Validation
    model.eval()
    y_true, y_pred, y_prob = [], [], []
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.float().to(device)
            logits = model(imgs)
            probs = torch.sigmoid(logits).cpu().numpy()
            y_prob.extend(probs)
            y_true.extend(labels.cpu().numpy())
            y_pred.extend((probs > 0.5).astype(int))
    metrics = classification_metrics(y_true, y_pred, y_prob)
    print(f"Val F1: {metrics['f1']:.4f}, ROC-AUC: {metrics['roc_auc']:.4f}")
    scheduler.step(metrics['f1'])

### Segmentation Training (Optional)

If you have segmentation masks, use the segmentation model and appropriate loss (e.g., BCE+Dice). The training loop is similar to the above.

In [ ]:
# Example: Segmentation training loop (if masks exist)
# from src.losses import bce_dice_loss
# optimizer = optim.AdamW(seg_model.parameters(), lr=1e-3, weight_decay=1e-4)
# for epoch in range(num_epochs):
#     seg_model.train()
#     for imgs, masks in tqdm(seg_loader, desc=f"Epoch {epoch+1}"):
#         imgs, masks = imgs.to(device), masks.to(device)
#         optimizer.zero_grad()
#         logits = seg_model(imgs)
#         loss = bce_dice_loss(logits, masks)
#         loss.backward()
#         optimizer.step()
#     print(f"Epoch {epoch+1} done")

## 6. Module 4: Model Evaluation

Evaluation is performed using metrics appropriate for each task:
- **Classification**: F1 score, ROC-AUC, confusion matrix, etc.
- **Segmentation**: Dice coefficient, IoU, pixel accuracy.

The metrics module provides easy-to-use functions for these evaluations.

In [ ]:
# Example: Evaluate classification model on test set
model.eval()
y_true, y_pred, y_prob = [], [], []
with torch.no_grad():
    for imgs, labels in test_loader:
        imgs, labels = imgs.to(device), labels.float().to(device)
        logits = model(imgs)
        probs = torch.sigmoid(logits).cpu().numpy()
        y_prob.extend(probs)
        y_true.extend(labels.cpu().numpy())
        y_pred.extend((probs > 0.5).astype(int))
metrics = classification_metrics(y_true, y_pred, y_prob)
print("Test Metrics:", metrics)

# Confusion matrix
cm = confusion_matrix(y_true, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix')
plt.show()

In [ ]:
# Example: Evaluate segmentation model (if masks exist)
# from src.metrics import dice_coef, iou_coef
# seg_model.eval()
# dices, ious = [], []
# with torch.no_grad():
#     for imgs, masks in seg_loader:
#         imgs, masks = imgs.to(device), masks.to(device)
#         logits = seg_model(imgs)
#         dices.append(dice_coef(logits, masks))
#         ious.append(iou_coef(logits, masks))
# print(f"Segmentation Dice: {np.mean(dices):.4f}, IoU: {np.mean(ious):.4f}")

## 7. Module 5: Model Inference and Prediction

After training, the model can be used to make predictions on new, unseen data. This section demonstrates how to load a trained model and perform inference, visualizing the results for both classification and segmentation tasks.

In [ ]:
# Example: Inference for classification
from PIL import Image

def predict_image(img_path, model, transform, device):
    model.eval()
    img = Image.open(img_path).convert('RGB')
    x = transform(img).unsqueeze(0).to(device)
    with torch.no_grad():
        logits = model(x)
        prob = torch.sigmoid(logits).item()
        pred = int(prob > 0.5)
    return pred, prob

# Example usage:
# img_path = test_files[0]
# pred, prob = predict_image(img_path, model, val_transform, device)
# print(f"Predicted: {pred}, Probability: {prob:.3f}")

In [ ]:
# Example: Inference for segmentation (if masks exist)
# def predict_mask(img_path, model, transform, device):
#     model.eval()
#     img = Image.open(img_path).convert('RGB')
#     x = transform(img=np.array(img))['image'].unsqueeze(0).to(device)
#     with torch.no_grad():
#         logits = model(x)
#         mask_pred = torch.sigmoid(logits).squeeze().cpu().numpy()
#     return mask_pred
#
# # Example usage:
# # img_path = img_paths[0]
# # mask_pred = predict_mask(img_path, seg_model, seg_transform, device)
# # plt.imshow(mask_pred > 0.5, cmap='gray')
# # plt.title('Predicted Mask')
# # plt.show()